# Phase 5 - Legal Statute Identification (LSI) with Arch_2 + PEFT

Multi-label statute prediction starting from the **Phase-1 Arch_2 (local/global hybrid decoder)** checkpoint. No knowledge distillation.

Arch_2 is a causal decoder with sliding-window attention layers (window 128 -> 512 with depth) and every 4th layer global (NoPE).
The same per-layer window masks as in Phase 1 are used. Every window is mean-pooled over its real tokens (after the final RMSNorm).

**What is in here**
- Tokenizer + model code: copied **unchanged** from the Phase-1 notebook.
- PEFT written from scratch for the model's `nn.Linear` layers: **soft prompt**, **bottleneck adapters**, **LoRA**, **LoRMA** (+ `head_only` baseline).
- Long documents: **sliding windows** (`CHUNK_LEN` tokens, step `CHUNK_STRIDE`, so windows overlap). The last window ends at the last token of the document.
- Window -> document: **per-statute attention** (`label_attn`). Each statute gets its own attention over the windows, so it can focus on the window that justifies it.
- Multi-label head (`Linear` + sigmoid, BCE with `pos_weight`) and one global threshold tuned on dev.
- **Works on one or two GPUs** (Kaggle T4 x2): the windows of a batch are split over both GPUs with `DataParallel`.
- Output: `prediction_<method>.json` in the format `[{"id": ..., "pred": [ints]}]`.

**Before running:** attach the `kratikasum/tokenizer` dataset, attach the dataset with the Arch_2 checkpoint (`best.pt` or `last.pt`, or set `CKPT_PATH`),
and add the Hugging Face token as Kaggle secret `HF`. Choose the method with `METHOD` in the config cell.

## 1. Imports and configuration

In [ ]:
import os, random, json, time, math, zipfile, gc, glob, pickle
from array import array
from dataclasses import asdict
from typing import List, Dict, Tuple
import numpy as np
import torch
from huggingface_hub import login, hf_hub_download
from kaggle_secrets import UserSecretsClient

HF_TOKEN = UserSecretsClient().get_secret("HF")
os.environ["HF_TOKEN"] = HF_TOKEN
login(token=HF_TOKEN)

print('CUDA available:', torch.cuda.is_available())
for i in range(torch.cuda.device_count()):
    print(f'GPU {i}:', torch.cuda.get_device_name(i))

In [ ]:
import torch.nn as nn
import torch.nn.functional as F
from sklearn.metrics import f1_score
from torch.utils.data import DataLoader
from tqdm.auto import tqdm

# ---------------- data ----------------
HF_DATASET_REPO = 'Exploration-Lab/CS-781-Capstone'
TRAIN_FILE = 'Phase_5_LSI/Train/train.json'
DEV_FILE   = 'Phase_5_LSI/Train/dev.json'
TEST_FILE  = 'Phase_5_LSI/Test/test.json'

# ---------------- Phase-1 model ----------------
CKPT_PATH = None            # path to the Arch_2 checkpoint (best.pt or last.pt); None -> search /kaggle/input and /kaggle/working
GRAD_CKPT = True            # gradient checkpointing (saves memory)

# ---------------- GPUs ----------------
N_GPU = torch.cuda.device_count()   # Kaggle T4 x2 -> 2: the windows of every batch are split across both GPUs

# ---------------- long documents (sliding windows) ----------------
CHUNK_LEN    = 512          # window length in tokens
CHUNK_STRIDE = 384          # step between windows (384 -> 128 tokens of overlap)
MAX_CHUNKS   = 6            # windows kept per document
CHUNK_SELECT = 'even'       # if a document has more windows: 'even' (evenly spaced) | 'head_tail' (start + end) | 'relevance' (most statute-like)
ATTN_DIM     = 128          # hidden size of the per-statute window attention

# ---------------- training ----------------
SEED = 42
EPOCHS = 6
DOCS_PER_GPU = 2                        # documents per GPU per step (lower this if you get CUDA OOM)
BATCH_DOCS = DOCS_PER_GPU * max(1, N_GPU)
GRAD_ACCUM = max(1, 8 // BATCH_DOCS)    # effective batch = 8 documents on 1 or 2 GPUs
WARMUP_FRAC = 0.1
WEIGHT_DECAY = 0.01
GRAD_CLIP = 1.0
HEAD_LR = 1e-3              # classification head
PEFT_LR = {'soft_prompt': 3e-3, 'adapter': 1e-3, 'lora': 3e-4, 'lorma': 3e-4, 'head_only': 0.0}
USE_POS_WEIGHT = True       # up-weight rare statutes in the BCE loss
LOSS = 'bce'                # 'bce' or 'focal'

# ---------------- PEFT ----------------
# ONE method per run: 'soft_prompt' | 'adapter' | 'lora' | 'lorma' | 'head_only' (baseline), or 'all' for the four PEFT methods
METHOD = 'lora'
METHODS = ['soft_prompt', 'adapter', 'lora', 'lorma'] if METHOD == 'all' else [METHOD]
PEFT = dict(
    num_virtual_tokens=16,
    init_text=' Identify the statutes relevant to this legal case:',
    bottleneck=16,                       # adapter size
    rank=8, alpha=16,                    # LoRA / LoRMA
    targets=('q', 'k', 'v', 'o'),        # attention projections that get LoRA / LoRMA
)

OUTPUT_DIR = '/kaggle/working/lsi_arch2'

In [ ]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'
# T4/P100 have no bf16 -> fp16 + GradScaler; Ampere+ -> bf16
use_bf16 = device == 'cuda' and torch.cuda.get_device_capability()[0] >= 8
amp_dtype = torch.bfloat16 if use_bf16 else torch.float16
print('device =', device, '| GPUs used =', max(1, N_GPU) if device == 'cuda' else 0, '| amp dtype =', amp_dtype)

## 2. Tokenizer (unchanged from Phase 1)

In [ ]:
class ByteLevelBPE:
    """Your tokenizer class (training code removed: we load the already-trained pickle)."""
    def __init__(self):
        self.vocab: Dict[bytes, int] = {}
        self.inverse_vocab: Dict[int, bytes] = {}
        self.merges: Dict[Tuple[bytes, bytes], Tuple[bytes, int]] = {}
        self.word_end_marker = b'</w>'
        self.reserved_tokens = [b'<pad>', b'<unk>', b'<bos>', b'<eos>']

    def tokenize(self, text: str) -> List[bytes]:
        words = text.split()
        tokens = []
        for w in words:
            word_bytes = w.encode('utf-8')
            split = [bytes([b]) for b in word_bytes] + [self.word_end_marker]
            while True:
                best_pair, best_rank = None, float('inf')
                for i in range(len(split) - 1):
                    pair = (split[i], split[i+1])
                    if pair in self.merges:
                        rank = self.merges[pair][1]
                        if rank < best_rank:
                            best_rank, best_pair = rank, pair
                if best_pair is None:
                    break
                new_split, i = [], 0
                while i < len(split):
                    if i < len(split) - 1 and split[i] == best_pair[0] and split[i+1] == best_pair[1]:
                        new_split.append(self.merges[best_pair][0])
                        i += 2
                    else:
                        new_split.append(split[i])
                        i += 1
                split = new_split
            tokens.extend(split)
        return tokens

    def encode(self, text: str) -> List[int]:
        unk_id = self.vocab[b'<unk>']
        return [self.vocab.get(token, unk_id) for token in self.tokenize(text)]

    def decode(self, token_ids: List[int]) -> str:
        skip_ids = {self.vocab[t] for t in self.reserved_tokens if t in self.vocab}
        bytes_list = []
        for tid in token_ids:
            if tid in skip_ids:
                continue
            bytes_list.append(self.inverse_vocab.get(tid, b''))
        joined = b''.join(bytes_list)
        return joined.replace(self.word_end_marker, b' ').decode('utf-8', errors='replace').strip()

    @classmethod
    def load(cls, filepath: str):
        with open(filepath, 'rb') as f:
            data = pickle.load(f)
        bpe = cls()
        bpe.vocab = data['vocab']
        bpe.inverse_vocab = {v: k for k, v in bpe.vocab.items()}
        bpe.merges = data['merges']
        return bpe

    def __len__(self):
        return len(self.vocab)

    @property
    def eos_token_id(self):
        return self.vocab.get(b'<eos>')

In [ ]:
cands = glob.glob('/kaggle/input/**/bpe_tokenized_txt.pkl', recursive=True)
if cands:
    tok_path = cands[0]
else:                                   # dataset not attached as input -> download it
    import kagglehub
    tok_path = os.path.join(kagglehub.dataset_download('kratikasum/tokenizer'), 'bpe_tokenized_txt.pkl')
tokenizer = ByteLevelBPE.load(tok_path)

# The embedding table must cover the largest id, and ids should be unique.
VOCAB_SIZE = max(tokenizer.vocab.values()) + 1
n_unique = len(set(tokenizer.vocab.values()))
print(f'tokenizer entries: {len(tokenizer):,} | unique ids: {n_unique:,} | embedding size: {VOCAB_SIZE:,}')
if n_unique != len(tokenizer):
    print('WARNING: some tokens share an id (can happen when two merges build the same byte string).')

In [ ]:
word_cache = {}                          # tokenizer works word by word, so caching per word is exact

def fast_encode(text):
    out = []
    for word in text.split():
        if word not in word_cache:
            word_cache[word] = tokenizer.encode(word)
        out.extend(word_cache[word])
    return out

## 3. Arch_2 model code (unchanged from Phase 1)

In [ ]:
import math
from dataclasses import dataclass
from typing import List, Optional
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.checkpoint import checkpoint

In [ ]:
@dataclass
class Config:
    vocab_size: int = 32000
    d_model: int = 768
    n_layer: int = 12
    n_head: int = 12
    n_kv_head: Optional[int] = None
    max_seq_len: int = 2048
    rope_theta: float = 10000.0
    rope_pct: float = 1.0
    dropout: float = 0.0
    tie_embeddings: bool = False
    qk_norm: bool = False
    grad_ckpt: bool = False
    layer_types: Optional[List[str]] = None   # per layer: "l<W>" sliding window W + RoPE, "n" global NoPE
    mtp_heads: int = 0
    mtp_weight: float = 0.15

    @property
    def head_dim(self):
        return self.d_model // self.n_head

    @property
    def kv_heads(self):
        return self.n_kv_head or self.n_head

    @property
    def ff_dim(self):
        return int(round(8 / 3 * self.d_model / 64)) * 64

In [ ]:
class RMSNorm(nn.Module):
    def __init__(self, d, eps=1e-6):
        super().__init__()
        self.w = nn.Parameter(torch.ones(d))
        self.eps = eps

    def forward(self, x):
        xf = x.float()
        out = xf * torch.rsqrt(xf.pow(2).mean(-1, keepdim=True) + self.eps)
        return (out * self.w.float()).type_as(x)


class Rotary(nn.Module):
    def __init__(self, dim, max_len, theta):
        super().__init__()
        inv = 1.0 / (theta ** (torch.arange(0, dim, 2).float() / max(dim, 1)))
        freqs = torch.outer(torch.arange(max_len).float(), inv)
        self.register_buffer("cos", freqs.cos(), persistent=False)
        self.register_buffer("sin", freqs.sin(), persistent=False)

    def forward(self, T, start=0):
        """cos/sin for positions start .. start+T-1 (start > 0 only while generating with a cache)."""
        return self.cos[start:start + T], self.sin[start:start + T]


def apply_rope(x, cos, sin, rot_dim):
    """x: (B,H,T,hd). Rotates the first rot_dim dims (half-split convention)."""
    if rot_dim == 0:
        return x
    xr, xp = x[..., :rot_dim], x[..., rot_dim:]
    x1, x2 = xr.float().chunk(2, dim=-1)
    c, s = cos[None, None], sin[None, None]
    xr = torch.cat([x1 * c - x2 * s, x1 * s + x2 * c], dim=-1).to(x.dtype)
    return torch.cat([xr, xp], dim=-1)


class MLP(nn.Module):
    """SwiGLU feed-forward."""
    def __init__(self, cfg):
        super().__init__()
        d, f = cfg.d_model, cfg.ff_dim
        self.gate = nn.Linear(d, f, bias=False)
        self.up = nn.Linear(d, f, bias=False)
        self.down = nn.Linear(f, d, bias=False)

    def forward(self, x):
        return self.down(F.silu(self.gate(x)) * self.up(x))

In [ ]:
# --------------------------------------------------------------------------------------
# KV cache + attention
# --------------------------------------------------------------------------------------
class KVCache:
    """Pre-allocated key/value buffer for ONE attention layer (stored un-expanded, i.e. Hkv heads).
    Lets generation process one new token per step instead of re-running the whole prefix."""

    def __init__(self, max_len):
        self.max_len, self.n, self.k, self.v = max_len, 0, None, None

    def append(self, k, v):
        if self.k is None:
            B, H, _, hd = k.shape
            self.k = k.new_empty(B, H, self.max_len, hd)
            self.v = v.new_empty(B, H, self.max_len, hd)
        T = k.size(2)
        self.k[:, :, self.n:self.n + T] = k
        self.v[:, :, self.n:self.n + T] = v
        self.n += T

    def get(self, window=None):
        s = 0 if window is None else max(0, self.n - window)
        return self.k[:, :, s:self.n], self.v[:, :, s:self.n]


def sdpa_gqa(q, k, v, mask=None, causal=False, p=0.0):
    """q: (B,H,T,hd)   k,v: (B,Hkv,S,hd).  Handles GQA without copying the cache when T == 1."""
    B, H, T, hd = q.shape
    Hkv = k.size(1)
    if T == 1 and mask is None:                       # single-token decode step
        if Hkv != H:                                  # fold the query-head group into the "length" axis
            r = H // Hkv
            out = F.scaled_dot_product_attention(q.reshape(B, Hkv, r, hd), k, v)
            return out.reshape(B, H, 1, hd)
        return F.scaled_dot_product_attention(q, k, v)
    if Hkv != H:
        r = H // Hkv
        k, v = k.repeat_interleave(r, dim=1), v.repeat_interleave(r, dim=1)
    return F.scaled_dot_product_attention(q, k, v, attn_mask=mask, dropout_p=p,
                                          is_causal=causal and mask is None)

In [ ]:
class Attention(nn.Module):
    """MHA/GQA self-attention with optional RoPE (or NoPE), QK-norm, value residual, sliding-window mask
    and KV cache.  Returns (output, v_of_the_new_tokens)."""

    def __init__(self, cfg, causal=True, use_rope=True, value_residual=False):
        super().__init__()
        d = cfg.d_model
        self.H, self.Hkv, self.hd = cfg.n_head, cfg.kv_heads, cfg.head_dim
        assert self.H % self.Hkv == 0, "n_head must be divisible by n_kv_head"
        self.q = nn.Linear(d, self.H * self.hd, bias=False)
        self.k = nn.Linear(d, self.Hkv * self.hd, bias=False)
        self.v = nn.Linear(d, self.Hkv * self.hd, bias=False)
        self.o = nn.Linear(self.H * self.hd, d, bias=False)
        self.q_norm = RMSNorm(self.hd) if cfg.qk_norm else None
        self.k_norm = RMSNorm(self.hd) if cfg.qk_norm else None
        self.p_drop = cfg.dropout
        self.causal = causal
        self.rot_dim = (int(self.hd * cfg.rope_pct) // 2 * 2) if use_rope else 0
        self.lam = nn.Parameter(torch.zeros(self.Hkv, 1, 1)) if value_residual else None

    def forward(self, x, rope=None, v_first=None, mask=None, cache=None, window=None):
        B, T, _ = x.shape
        q = self.q(x).view(B, T, self.H, self.hd).transpose(1, 2)
        k = self.k(x).view(B, T, self.Hkv, self.hd).transpose(1, 2)
        v = self.v(x).view(B, T, self.Hkv, self.hd).transpose(1, 2)
        if self.lam is not None and v_first is not None:
            lam = torch.sigmoid(self.lam).to(v.dtype)
            v = (1 - lam) * v + lam * v_first
        v_new = v
        if self.q_norm is not None:
            q, k = self.q_norm(q), self.k_norm(k)
        if self.rot_dim:
            cos, sin = rope
            q, k = apply_rope(q, cos, sin, self.rot_dim), apply_rope(k, cos, sin, self.rot_dim)
        causal = self.causal
        if cache is not None:
            cache.append(k, v)
            k, v = cache.get(window if T == 1 else None)   # window only matters for 1-token steps
            causal = T > 1                                  # prefill: causal; decode step: sees everything cached
        out = sdpa_gqa(q, k, v, mask=mask, causal=causal, p=self.p_drop if self.training else 0.0)
        return self.o(out.transpose(1, 2).reshape(B, T, -1)), v_new

In [ ]:
class LMHead(nn.Module):
    """Final norm + unembedding (optionally tied) + optional multi-token-prediction heads."""

    def __init__(self, cfg, tok_emb: nn.Embedding):
        super().__init__()
        self.cfg = cfg
        self.norm = RMSNorm(cfg.d_model)
        self.proj = nn.Linear(cfg.d_model, cfg.vocab_size, bias=False)
        if cfg.tie_embeddings:
            self.proj.weight = tok_emb.weight
        self.mtp = nn.ModuleList([nn.Linear(cfg.d_model, cfg.d_model, bias=False)
                                  for _ in range(getattr(cfg, "mtp_heads", 0))])

    def logits(self, h):
        return self.proj(self.norm(h))

    def forward(self, h, targets=None):
        h = self.norm(h)
        logits = self.proj(h)
        if targets is None:
            return logits, None
        V = logits.size(-1)
        loss = F.cross_entropy(logits.reshape(-1, V).float(), targets.reshape(-1),
                               ignore_index=-100, label_smoothing=0.1)  # ← ADD label_smoothing
        self.main_loss = loss.detach()
        w = getattr(self.cfg, "mtp_weight", 0.3)
        for k, p in enumerate(self.mtp, start=1):
            if targets.size(1) <= k:
                break
            hk = h[:, :-k] + F.silu(p(h[:, :-k]))
            lk = self.proj(hk)
            loss = loss + w * F.cross_entropy(lk.reshape(-1, V).float(), targets[:, k:].reshape(-1),
                                              ignore_index=-100, label_smoothing=0.1)  # ← here too
        return logits, loss

In [ ]:
def ckpt_call(fn, enabled, *args):
    return checkpoint(fn, *args, use_reentrant=False) if enabled else fn(*args)


def init_weights(model, depth):
    for m in model.modules():
        if isinstance(m, (nn.Linear, nn.Embedding)):
            nn.init.normal_(m.weight, 0.0, 0.02)
    std = 0.02 / math.sqrt(2 * depth)
    for n, p in model.named_parameters():
        if n.endswith(("o.weight", "down.weight")):
            nn.init.normal_(p, 0.0, std)


class BaseLM(nn.Module):
    """Common interface used by training.py / generation.py:
         forward(x, targets=None, src=None) -> (logits, loss)
         prefill(prompt_ids, max_new)       -> (logits_for_next_token, state)      [generation]
         step(token_ids, state)             -> logits_for_next_token               [generation]
       `eval_offset` = number of leading label positions that are NOT scored (encoder-decoder only)."""
    eval_offset = 0

    def num_params(self, non_embedding=False):
        n = sum(p.numel() for p in self.parameters())
        return n - self.tok.weight.numel() if non_embedding else n

In [ ]:
def parse_layer(t: str):
    """returns (window, use_rope):  "n" -> global NoPE,  "l<W>" -> sliding window W + RoPE"""
    if t == "n":
        return None, False
    if t.startswith("l"):
        return int(t[1:]), True
    raise ValueError(f"unknown layer type {t!r}")


class Block(nn.Module):
    """Pre-norm residual block; `window` = sliding-window size (None = full causal attention)."""

    def __init__(self, cfg, window, use_rope):
        super().__init__()
        self.n1, self.n2 = RMSNorm(cfg.d_model), RMSNorm(cfg.d_model)
        self.attn, self.mlp = Attention(cfg, use_rope=use_rope), MLP(cfg)
        self.drop = nn.Dropout(cfg.dropout)
        self.window = window

    def forward(self, x, rope, mask=None, cache=None):
        h, _ = self.attn(self.n1(x), rope, mask=mask, cache=cache, window=self.window)
        x = x + self.drop(h)
        return x + self.drop(self.mlp(self.n2(x)))

In [ ]:
class DecoderLM(BaseLM):
    def __init__(self, cfg: Config):
        super().__init__()
        self.cfg = cfg
        assert len(cfg.layer_types) == cfg.n_layer, "layer_types must have n_layer entries"
        self.tok = nn.Embedding(cfg.vocab_size, cfg.d_model)
        self.rope = Rotary(int(cfg.head_dim * cfg.rope_pct) // 2 * 2, cfg.max_seq_len, cfg.rope_theta)
        self.blocks = nn.ModuleList([Block(cfg, *parse_layer(t)) for t in cfg.layer_types])
        self.head = LMHead(cfg, self.tok)
        init_weights(self, cfg.n_layer)

    def hidden(self, idx, caches=None, start=0):
        cfg = self.cfg
        B, T = idx.shape
        x = self.tok(idx)
        rope = self.rope(T, start)
        masks = {}
        for i, blk in enumerate(self.blocks):
            mask = None
            if T > 1 and blk.window is not None and blk.window < T:     # dense band mask (training / prefill)
                if blk.window not in masks:
                    r = torch.arange(T, device=idx.device)
                    d = r[:, None] - r[None, :]
                    masks[blk.window] = (d >= 0) & (d < blk.window)
                mask = masks[blk.window]
            if caches is None:
                x = ckpt_call(blk, cfg.grad_ckpt and self.training, x, rope, mask)
            else:
                x = blk(x, rope, mask, cache=caches[i])
        return x

    def forward(self, idx, targets=None, src=None):
        return self.head(self.hidden(idx), targets)

    # ---- generation (KV cache; local layers only look at their last `window` keys) ----
    def prefill(self, idx, max_new):
        caches = [KVCache(idx.shape[1] + max_new) for _ in self.blocks]
        h = self.hidden(idx, caches=caches)
        return self.head.logits(h[:, -1]), caches

    def step(self, tok, caches):
        h = self.hidden(tok, caches=caches, start=caches[0].n)
        return self.head.logits(h[:, -1])

In [ ]:
Model = DecoderLM


BASE = dict(d_model=768, n_layer=12, n_head=12, max_seq_len=2048)


def hybrid_pattern(n_layer, global_every=4, w_min=128, w_max=512):
    n_local = n_layer - n_layer // global_every if global_every else n_layer
    types, li = [], 0
    for i in range(n_layer):
        if global_every and (i + 1) % global_every == 0:
            types.append("n")
        else:
            f = li / max(1, n_local - 1)
            w = int(2 ** round(math.log2(w_min) + f * (math.log2(w_max) - math.log2(w_min))))
            types.append(f"l{w}")
            li += 1
    return types


def cfg_baseline(**kw):
    cfg = Config(**{**BASE, **kw})
    if cfg.layer_types is None:
        cfg.layer_types = [f"l{min(512, cfg.max_seq_len)}"] * cfg.n_layer
    return cfg


def cfg_modified(global_every=4, w_min=128, w_max=512, **kw):
    cfg = Config(**{**BASE, "n_layer": 17, "n_kv_head": 4, "qk_norm": True, "tie_embeddings": True, **kw})
    cfg.mtp_heads = 2
    cfg.mtp_weight = 0.15
    if cfg.layer_types is None:
        cfg.layer_types = hybrid_pattern(cfg.n_layer, global_every, w_min, w_max)
    return cfg

## 4. Load the Arch_2 checkpoint
`best.pt` / `last.pt` stores the Phase-1 config, so the exact same architecture is rebuilt.

In [ ]:
if CKPT_PATH is None:
    found = [p for name in ('best.pt', 'last.pt')
             for p in glob.glob(f'/kaggle/input/**/{name}', recursive=True) + glob.glob(f'/kaggle/working/**/{name}', recursive=True)]
    assert found, 'checkpoint not found - set CKPT_PATH'
    print('checkpoints found:', found)
    CKPT_PATH = found[0]
print('Arch_2 checkpoint:', CKPT_PATH)
BASE_CKPT = torch.load(CKPT_PATH, map_location='cpu', weights_only=False)   # stays on CPU, reused for every method
cfg_saved = BASE_CKPT['cfg']
assert 'layer_types' in cfg_saved and 'la_decay' not in cfg_saved, 'this checkpoint is not Arch_2 - set CKPT_PATH to the right file'

def load_backbone():
    """Fresh copy of the Phase-1 model (every experiment starts from the pretrained weights)."""
    cfg = Config(**cfg_saved)
    cfg.grad_ckpt = GRAD_CKPT
    m = Model(cfg)
    m.load_state_dict(BASE_CKPT['model'])
    return m.to(device)

## 5. Data
Labels are statute IDs. The label space is every ID seen in train + dev; each document becomes a multi-hot vector.
Each document is tokenized with the Phase-1 tokenizer and cut into **overlapping sliding windows** (`CHUNK_LEN` tokens, step `CHUNK_STRIDE`).
If a document has more than `MAX_CHUNKS` windows, `CHUNK_SELECT` chooses which ones to keep.

In [ ]:
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

def load_split(fname):
    path = hf_hub_download(repo_id=HF_DATASET_REPO, filename=fname, repo_type='dataset', token=HF_TOKEN)
    raw = json.load(open(path, encoding='utf-8'))
    if isinstance(raw, dict):                                   # {id: {...}} -> list of dicts
        raw = [{'id': k, **v} for k, v in raw.items()]
    return raw

train_raw, dev_raw, test_raw = load_split(TRAIN_FILE), load_split(DEV_FILE), load_split(TEST_FILE)
print('train / dev / test documents:', len(train_raw), len(dev_raw), len(test_raw))

# label space = every statute id seen in train + dev
label_ids = sorted({int(l) for d in train_raw + dev_raw for l in d['labels']})
label_to_col = {l: j for j, l in enumerate(label_ids)}
N_LABELS = len(label_ids)
print('number of statutes:', N_LABELS)

PAD_ID = tokenizer.vocab[b'<pad>']

def multi_hot(labels):
    y = np.zeros(N_LABELS, dtype=np.float32)
    for l in labels:
        y[label_to_col[int(l)]] = 1.0
    return y

def window_starts(n_tokens, length, stride):
    """Start of every overlapping window. The last window is moved so it ends at the last token."""
    if n_tokens <= length:
        return [0]
    starts = list(range(0, n_tokens - length + 1, stride))
    if starts[-1] + length < n_tokens:
        if n_tokens - length - starts[-1] < stride // 2:
            starts[-1] = n_tokens - length          # nearly the same window: slide it to the end
        else:
            starts.append(n_tokens - length)
    return starts

_REL = {}
def get_token_relevance():
    """How statute-specific each token id is: log P(token | statute) - log P(token), from TRAIN documents only."""
    if 'tok' in _REL:
        return _REL['tok']
    N = len(train_raw)
    C = np.zeros((N_LABELS, VOCAB_SIZE), dtype=np.float32); n_l = np.zeros(N_LABELS, dtype=np.float32)
    df = np.zeros(VOCAB_SIZE, dtype=np.float32)
    for d in tqdm(train_raw, desc='relevance statistics', leave=False):
        u = np.unique(np.asarray(fast_encode(d['text']), dtype=np.int64))
        df[u] += 1
        for l in d['labels']:
            c = label_to_col[int(l)]
            C[c, u] += 1; n_l[c] += 1
    lift = np.log((C + 1) / (n_l[:, None] + 2)) - np.log((df + 1) / (N + 2))[None, :]
    lift[C < 2] = 0
    _REL['tok'] = np.clip(lift, 0, None).max(0).astype(np.float32)        # best score over all statutes
    return _REL['tok']

def select_windows(ids, starts, k):
    """Indices of the k windows to keep (in document order)."""
    n = len(starts)
    if n <= k:
        return list(range(n))
    if CHUNK_SELECT == 'head_tail':                  # first half + last half of the budget
        n_head = (k + 1) // 2
        return sorted(set(range(n_head)) | set(range(n - (k - n_head), n)))
    if CHUNK_SELECT == 'relevance':                  # windows whose tokens are most statute-like
        tok = get_token_relevance()
        scores = np.array([tok[np.asarray(ids[s:s + CHUNK_LEN], dtype=np.int64)].mean() for s in starts])
        return sorted(np.argsort(-scores, kind='stable')[:k].tolist())
    return np.unique(np.linspace(0, n - 1, k).round().astype(int)).tolist()          # 'even'

def doc_chunks(text):
    """Document -> (list of token-id windows, list of window start offsets)."""
    ids = fast_encode(text) or [PAD_ID]
    starts = window_starts(len(ids), CHUNK_LEN, min(CHUNK_STRIDE, CHUNK_LEN))
    if len(starts) > MAX_CHUNKS:
        starts = [starts[i] for i in select_windows(ids, starts, MAX_CHUNKS)]
    return [ids[s:s + CHUNK_LEN] for s in starts], starts

def build(raw, has_labels=True):
    out = []
    for d in tqdm(raw, desc='tokenizing', leave=False):
        chunks, starts = doc_chunks(d['text'])
        out.append({'id': d['id'], 'chunks': chunks, 'starts': starts,
                    'y': multi_hot(d['labels']) if has_labels else np.zeros(N_LABELS, dtype=np.float32)})
    return out

train_ds, dev_ds, test_ds = build(train_raw), build(dev_raw), build(test_raw, has_labels=False)

def collate(batch):
    """All windows of the batch go into one padded tensor; n_chunks tells which windows belong to which document."""
    chunks = [c for ex in batch for c in ex['chunks']]
    L = max(len(c) for c in chunks)
    ids = torch.full((len(chunks), L), PAD_ID, dtype=torch.long)
    mask = torch.zeros(len(chunks), L, dtype=torch.bool)
    for i, c in enumerate(chunks):
        ids[i, :len(c)] = torch.tensor(c)
        mask[i, :len(c)] = True
    y = torch.from_numpy(np.stack([ex['y'] for ex in batch]))
    return ids, mask, [len(ex['chunks']) for ex in batch], y

train_loader = DataLoader(train_ds, batch_size=BATCH_DOCS, shuffle=True, collate_fn=collate)
dev_loader   = DataLoader(dev_ds,   batch_size=BATCH_DOCS, shuffle=False, collate_fn=collate)
test_loader  = DataLoader(test_ds,  batch_size=BATCH_DOCS, shuffle=False, collate_fn=collate)

# pos_weight for BCE: rare statutes get a larger weight (sqrt softens it, capped at 10)
pos = np.stack([ex['y'] for ex in train_ds]).sum(0).clip(min=1)
POS_WEIGHT = torch.tensor(np.sqrt((len(train_ds) - pos).clip(min=1) / pos).clip(1, 10), dtype=torch.float32)
print('avg labels per document:', np.mean([ex['y'].sum() for ex in train_ds]).round(2))
n_win = np.array([len(ex['chunks']) for ex in train_ds])
print(f'windows/doc (train): mean {n_win.mean():.2f} | median {np.median(n_win):.0f} | max {n_win.max()} | '
      f'capped at MAX_CHUNKS={MAX_CHUNKS}: {(n_win == MAX_CHUNKS).mean():.1%} of docs')

## 6. PEFT modules

| Method | Where trainable parameters go |
|---|---|
| Soft prompt | 16 learned vectors put in front of the input embeddings |
| Adapter | small residual bottleneck after every transformer block |
| LoRA | `W x + (alpha/r) B A x` on the attention q/k/v/o |
| LoRMA | multiplicative: `y = Wx + s B A (Wx)` on the attention q/k/v/o |

All of them start as an exact identity (zero-initialised up / B matrix), so training starts from the pretrained model.

In [ ]:
class SoftPrompt(nn.Module):
    """Trainable vectors put in front of the token embeddings (initialised from the embeddings of a short text)."""
    def __init__(self, init_embeddings):
        super().__init__()
        self.prompt = nn.Parameter(init_embeddings.detach().clone().float())

    @property
    def n(self):
        return self.prompt.shape[0]

    def forward(self, x):                                     # [B, T, d] -> [B, n+T, d]
        p = self.prompt.to(x.dtype).unsqueeze(0).expand(x.size(0), -1, -1)
        return torch.cat([p, x], dim=1)


class BottleneckAdapter(nn.Module):
    """h + W_up GELU(W_down h). W_up starts at zero, so the adapter starts as the identity."""
    def __init__(self, d, bottleneck=16):
        super().__init__()
        self.down, self.up = nn.Linear(d, bottleneck), nn.Linear(bottleneck, d)
        nn.init.zeros_(self.up.weight); nn.init.zeros_(self.up.bias)

    def forward(self, h):
        return h + self.up(F.gelu(self.down(h)))


class LoRALinear(nn.Module):
    """Frozen linear layer + low-rank update:  y = W x + (alpha/r) * B (A x).  B starts at zero."""
    def __init__(self, base, rank=8, alpha=16):
        super().__init__()
        self.base = base
        self.scaling = alpha / rank
        self.lora_A = nn.Parameter(torch.empty(rank, base.in_features))
        self.lora_B = nn.Parameter(torch.zeros(base.out_features, rank))
        nn.init.kaiming_uniform_(self.lora_A, a=math.sqrt(5))

    def forward(self, x):
        return self.base(x) + self.scaling * F.linear(F.linear(x, self.lora_A), self.lora_B)


class LoRMALinear(nn.Module):
    """LoRMA (multiplicative):  W' = (s B A + I) W,  computed as  h = W x;  y = h + s B (A h).  B starts at zero."""
    def __init__(self, base, rank=8, alpha=16):
        super().__init__()
        self.base = base
        self.scaling = alpha / rank
        d_out = base.out_features
        self.lorma_A = nn.Parameter((torch.rand(rank, d_out) * 2 - 1) / math.sqrt(d_out))
        self.lorma_B = nn.Parameter(torch.zeros(d_out, rank))

    def forward(self, x):
        h = self.base(x)
        return h + self.scaling * F.linear(F.linear(h, self.lorma_A), self.lorma_B)


def attach_peft(backbone, method, p=PEFT):
    """Freeze the backbone, then add the trainable PEFT parameters.
    Returns (soft_prompt, adapters); each is None if the method does not use it."""
    for q in backbone.parameters():
        q.requires_grad_(False)
    d = backbone.cfg.d_model
    blocks = backbone.blocks
    soft, adapters = None, None

    if method == 'soft_prompt':
        ids = fast_encode(p['init_text']) or [PAD_ID]
        ids = [ids[i % len(ids)] for i in range(p['num_virtual_tokens'])]        # repeat the init text up to the prompt length
        soft = SoftPrompt(backbone.tok.weight[torch.tensor(ids, device=backbone.tok.weight.device)])
    elif method == 'adapter':
        adapters = nn.ModuleList([BottleneckAdapter(d, p['bottleneck']) for _ in blocks])    # one adapter after every block
    elif method in ('lora', 'lorma'):
        Wrap = LoRALinear if method == 'lora' else LoRMALinear
        for blk in blocks:
            attn = blk.attn
            for name in p['targets']:
                setattr(attn, name, Wrap(getattr(attn, name), p['rank'], p['alpha']))
    elif method != 'head_only':
        raise ValueError(f'unknown method {method}')
    return soft, adapters

## 7. Classifier: Arch_2 -> sliding windows -> per-statute attention -> multi-label head

Arch_2 is causal, so right-padding never changes the real tokens. The classifier re-uses the model's own blocks and the same sliding-window masks
as `DecoderLM.hidden()`. Each overlapping window is mean-pooled over its real tokens (after the final RMSNorm). `LabelAttnHead` then combines the
windows of a document: **every statute has its own attention over the windows**.

In [ ]:
def backbone_hidden(model, ids, mask, soft=None, adapters=None):
    """Arch_2 local/global decoder -> hidden states of the text tokens [N, T, d] after the final RMSNorm.
    Same computation as DecoderLM.hidden(), plus the optional soft prompt / adapters.
    `mask` is not needed here: the model is causal, so right-padding never changes the real tokens."""
    cfg = model.cfg
    x = model.tok(ids)
    n_soft = 0
    if soft is not None:
        x, n_soft = soft(x), soft.n
    T = x.size(1)
    rope = model.rope(T)
    band_masks = {}                                                  # one sliding-window mask per window size
    for i, blk in enumerate(model.blocks):
        attn_mask = None
        if blk.window is not None and blk.window < T:                # local layer: token sees only its last `window` tokens
            if blk.window not in band_masks:
                r = torch.arange(T, device=ids.device)
                dist = r[:, None] - r[None, :]
                band_masks[blk.window] = (dist >= 0) & (dist < blk.window)
            attn_mask = band_masks[blk.window]
        x = ckpt_call(blk, cfg.grad_ckpt and torch.is_grad_enabled(), x, rope, attn_mask)
        if adapters is not None:
            x = adapters[i](x)
    return model.head.norm(x[:, n_soft:])

In [ ]:
class Encoder(nn.Module):
    """Backbone (+ soft prompt / adapters) -> one mean-pooled vector per window.
    This is the part that DataParallel splits across the GPUs."""
    def __init__(self, backbone, soft=None, adapters=None):
        super().__init__()
        self.backbone, self.soft, self.adapters = backbone, soft, adapters

    def forward(self, ids, mask):
        # autocast must be inside forward() because DataParallel runs every GPU in its own thread
        with torch.autocast(device_type=device, dtype=amp_dtype, enabled=(device == 'cuda')):
            h = backbone_hidden(self.backbone, ids, mask, self.soft, self.adapters)
        h = h.float()
        m = mask.unsqueeze(-1).float()
        return (h * m).sum(1) / m.sum(1).clamp(min=1)         # mean over the real tokens of each window


class LabelAttnHead(nn.Module):
    """Window vectors of one document -> one logit per statute.
    Every statute has its own attention over the windows: logit_l = sum_w attn_wl * logit_wl."""
    def __init__(self, d, n_labels, attn_dim=128, dropout=0.1):
        super().__init__()
        self.drop = nn.Dropout(dropout)
        self.fc = nn.Linear(d, n_labels)                       # window -> statute logits
        self.att_v = nn.Linear(d, attn_dim)
        self.att_w = nn.Linear(attn_dim, n_labels, bias=False)
        nn.init.zeros_(self.att_w.weight)                      # start with equal attention on every window

    def forward(self, window_vec, n_chunks, return_attn=False):
        """window_vec [total_windows, d]; n_chunks[i] = number of windows of document i."""
        logits, attns = [], []
        for w in torch.split(window_vec, n_chunks):            # w: [n_windows, d] (one document)
            w = self.drop(w)
            attn = torch.softmax(self.att_w(torch.tanh(self.att_v(w))), dim=0)    # [n_windows, n_labels], sums to 1 over windows
            logits.append((attn * self.fc(w)).sum(0))
            attns.append(attn)
        logits = torch.stack(logits)
        return (logits, attns) if return_attn else logits


class LSIClassifier(nn.Module):
    def __init__(self, encoder, n_labels, dropout=0.1):
        super().__init__()
        self.encoder = encoder
        self.head = LabelAttnHead(encoder.backbone.cfg.d_model, n_labels, ATTN_DIM, dropout)
        # DataParallel is kept in a list so it is not registered a second time (keeps the state_dict clean)
        self.run_encoder = [nn.DataParallel(encoder) if N_GPU > 1 else encoder]

    def train(self, mode=True):
        super().train(mode)
        self.encoder.backbone.eval()                           # the pretrained backbone always stays in eval mode
        return self

    def forward(self, ids, mask, n_chunks, return_attn=False):
        window_vec = self.run_encoder[0](ids, mask)            # [total_windows, d], windows split over the GPUs
        return self.head(window_vec, n_chunks, return_attn)    # head runs in fp32 on GPU 0

## 8. Thresholding, metrics, training and prediction
One global threshold is chosen on dev (micro-F1). Every document gets at least one statute (its highest-scoring one) if nothing passes the threshold.

In [ ]:
def decide(probs, thr):
    """probs [N, L] -> binary [N, L]; a row with nothing above thr gets its top statute."""
    pred = probs >= thr
    empty = ~pred.any(1)
    pred[empty, probs[empty].argmax(1)] = True
    return pred

def micro_f1(y, pred):
    tp = (y * pred).sum(); p, t = pred.sum(), y.sum()
    return 0.0 if tp == 0 else 2 * tp / (p + t)

def best_threshold(probs, y):
    grid = np.arange(0.05, 0.95, 0.025)
    scores = [micro_f1(y, decide(probs, t)) for t in grid]
    return float(grid[int(np.argmax(scores))]), float(max(scores))

def report(y, pred):
    return {'micro_f1': f1_score(y, pred, average='micro', zero_division=0),
            'macro_f1': f1_score(y, pred, average='macro', zero_division=0),
            'samples_f1': f1_score(y, pred, average='samples', zero_division=0)}

@torch.no_grad()
def predict_probs(clf, loader):
    clf.eval()
    out = []
    for ids, mask, n_chunks, _ in tqdm(loader, desc='predicting', leave=False):
        logits = clf(ids.to(device), mask.to(device), n_chunks)
        out.append(torch.sigmoid(logits.float()).cpu().numpy())
    return np.concatenate(out)

@torch.no_grad()
def collect_attention(clf, loader):
    """Window attention of every document: list of arrays [n_windows, n_labels] (float16)."""
    clf.eval(); out = []
    for ids, mask, n_chunks, _ in tqdm(loader, desc='attention', leave=False):
        _, attns = clf(ids.to(device), mask.to(device), n_chunks, return_attn=True)
        out += [a.float().cpu().numpy().astype(np.float16) for a in attns]
    return out

def trainable_state(clf):
    return {n: p.detach().cpu().clone() for n, p in clf.named_parameters() if p.requires_grad}

def multilabel_loss(logits, y, pos_weight=None):
    """BCE over all statutes (optionally with pos_weight), or focal BCE (gamma=2) if LOSS == 'focal'."""
    bce = F.binary_cross_entropy_with_logits(logits, y, pos_weight=pos_weight, reduction='none')
    if LOSS == 'focal':
        pt = torch.exp(-F.binary_cross_entropy_with_logits(logits, y, reduction='none'))
        bce = (1 - pt) ** 2 * bce
    return bce.mean()


def run_method(method):
    print(f'\n================ {method} ================')
    torch.manual_seed(SEED)
    backbone = load_backbone()                                    # start from the Phase-1 model every time
    soft, adapters = attach_peft(backbone, method)
    clf = LSIClassifier(Encoder(backbone, soft, adapters), N_LABELS).to(device)

    head_params = list(clf.head.parameters())
    peft_params = [p for n, p in clf.named_parameters() if p.requires_grad and not n.startswith('head.')]
    groups = [{'params': head_params, 'lr': HEAD_LR}]
    if peft_params:
        groups.append({'params': peft_params, 'lr': PEFT_LR[method]})
    n_peft = sum(p.numel() for p in peft_params)
    n_frozen = sum(p.numel() for p in clf.parameters() if not p.requires_grad)
    print(f'trainable: PEFT {n_peft:,} + head {sum(p.numel() for p in head_params):,} | frozen backbone {n_frozen:,}')

    opt = torch.optim.AdamW(groups, weight_decay=WEIGHT_DECAY)
    total_steps = EPOCHS * math.ceil(len(train_loader) / GRAD_ACCUM)
    warm = max(1, int(WARMUP_FRAC * total_steps))
    base_lrs = [g['lr'] for g in opt.param_groups]
    scaler = torch.amp.GradScaler('cuda', enabled=(device == 'cuda' and not use_bf16))
    pw = POS_WEIGHT.to(device) if USE_POS_WEIGHT else None

    y_dev = np.stack([ex['y'] for ex in dev_ds]).astype(bool)
    best = {'micro_f1': -1}
    step = 0
    for ep in range(EPOCHS):
        clf.train(); run, n = 0.0, 0
        opt.zero_grad(set_to_none=True)
        pbar = tqdm(train_loader, desc=f'{method} epoch {ep + 1}/{EPOCHS}')
        for it, (ids, mask, n_chunks, y) in enumerate(pbar):
            logits = clf(ids.to(device), mask.to(device), n_chunks)
            loss = multilabel_loss(logits.float(), y.to(device), pw)
            scaler.scale(loss / GRAD_ACCUM).backward()
            run += loss.item(); n += 1
            pbar.set_postfix(loss=f'{run / n:.4f}')
            if (it + 1) % GRAD_ACCUM == 0 or it + 1 == len(train_loader):
                # linear warm-up, then cosine decay
                f = (step + 1) / warm if step < warm else 0.5 * (1 + math.cos(math.pi * (step - warm) / max(1, total_steps - warm)))
                for g, b in zip(opt.param_groups, base_lrs):
                    g['lr'] = b * f
                scaler.unscale_(opt)
                torch.nn.utils.clip_grad_norm_([p for g in opt.param_groups for p in g['params']], GRAD_CLIP)
                scaler.step(opt); scaler.update()
                opt.zero_grad(set_to_none=True); step += 1

        # ---- dev evaluation, keep the best epoch ----
        probs = predict_probs(clf, dev_loader)
        thr, _ = best_threshold(probs, y_dev)
        m = report(y_dev, decide(probs, thr))
        tqdm.write(f'epoch {ep + 1}/{EPOCHS} | train loss {run / n:.4f} | dev micro-F1 {m["micro_f1"]:.4f} '
                   f'macro-F1 {m["macro_f1"]:.4f} samples-F1 {m["samples_f1"]:.4f} | thr {thr:.3f}')
        if m['micro_f1'] > best['micro_f1']:
            best = {**m, 'epoch': ep + 1, 'thr': thr, 'state': trainable_state(clf), 'dev_probs': probs}

    # ---- restore the best epoch and predict the test set ----
    clf.load_state_dict(best['state'], strict=False)
    test_probs = predict_probs(clf, test_loader)
    pred = decide(test_probs, best['thr'])
    submission = [{'id': ex['id'], 'pred': sorted({int(label_ids[j]) for j in np.where(pred[i])[0]})}
                  for i, ex in enumerate(test_ds)]
    best['test_probs'] = test_probs
    return clf, best, submission, n_peft

## 9. Run

In [ ]:
os.makedirs(OUTPUT_DIR, exist_ok=True)
for method in METHODS:
    RUN = method
    clf, best, submission, n_peft = run_method(method)
    with open(os.path.join(OUTPUT_DIR, f'prediction_{RUN}.json'), 'w') as f:
        json.dump(submission, f)
    torch.save({'state': best['state'], 'threshold': best['thr'], 'label_ids': label_ids, 'peft': PEFT, 'method': method},
               os.path.join(OUTPUT_DIR, f'weights_{RUN}.pt'))
    # result files are read again by the analysis below (also from earlier sessions)
    np.save(os.path.join(OUTPUT_DIR, f'dev_probs_{RUN}.npy'), best['dev_probs'])
    json.dump({'method': method, 'max_chunks': MAX_CHUNKS, 'chunk_stride': CHUNK_STRIDE, 'chunk_select': CHUNK_SELECT,
               'trainable_peft_params': n_peft, 'best_epoch': best['epoch'], 'threshold': best['thr'],
               'dev_micro_f1': best['micro_f1'], 'dev_macro_f1': best['macro_f1'], 'dev_samples_f1': best['samples_f1']},
              open(os.path.join(OUTPUT_DIR, f'results_{RUN}.json'), 'w'), indent=1)
    att = collect_attention(clf, dev_loader)                 # window attention on dev, used in 10.6
    pickle.dump({'attn': att, 'starts': [ex['starts'] for ex in dev_ds], 'ids': [ex['id'] for ex in dev_ds]},
                open(os.path.join(OUTPUT_DIR, f'dev_attn_{RUN}.pkl'), 'wb'))
    print(f'{RUN}: dev micro-F1 {best["micro_f1"]:.4f} | macro-F1 {best["macro_f1"]:.4f} | saved prediction_{RUN}.json')
    del clf; gc.collect(); torch.cuda.empty_cache()

## 10. Analysis
Everything below uses the dev set and the probabilities saved in section 9. It answers the questions the phase asks:
which adaptation works best (also for **rare statutes** and **multi-statute documents**), **global vs per-label thresholds**,
**precision/recall trade-off**, **calibration**, **label co-occurrence** and **where in the document the evidence is**.

### 10.0 Load results of every method that has been run
Reads every `results_<method>.json` and `dev_probs_<method>.npy` from the output folder, so methods trained in different sessions are compared together.

In [ ]:
results, dev_probs = {}, {}
for rp in sorted(glob.glob(os.path.join(OUTPUT_DIR, 'results_*.json'))):
    m = os.path.basename(rp)[len('results_'):-len('.json')]
    pp = os.path.join(OUTPUT_DIR, f'dev_probs_{m}.npy')
    if os.path.exists(pp):
        results[m], dev_probs[m] = json.load(open(rp)), np.load(pp)
assert results, 'no finished method found in OUTPUT_DIR - run section 9 first'
print('methods found:', list(results))
print(f'{"method":30s} {"PEFT params":>12s} {"best epoch":>10s}')
for k, r in results.items():
    print(f'{k:30s} {r["trainable_peft_params"]:12,d} {r["best_epoch"]:10d}')

### 10.1 Common helpers + overall comparison

In [ ]:
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

y_dev = np.stack([ex['y'] for ex in dev_ds]).astype(bool)
train_Y = np.stack([ex['y'] for ex in train_ds]).astype(bool)
train_freq = train_Y.sum(0)                        # how many training documents each statute has

def per_label_f1(y, pred):
    tp = (y & pred).sum(0); fp = (~y & pred).sum(0); fn = (y & ~pred).sum(0)
    den = 2 * tp + fp + fn
    return np.where(den > 0, 2 * tp / np.maximum(den, 1), np.nan), y.sum(0)    # nan = label absent in y and never predicted

def prf(y, pred):
    tp = (y & pred).sum(); p = tp / max(pred.sum(), 1); r = tp / max(y.sum(), 1)
    return p, r, (0 if tp == 0 else 2 * p * r / (p + r))

# overall table: micro / macro / samples F1 + precision / recall at each method's tuned threshold
print(f'{"method":30s} {"thr":>5s} {"P":>6s} {"R":>6s} {"micro-F1":>9s} {"macro-F1":>9s} {"samples-F1":>10s}')
for m, probs in dev_probs.items():
    pred = decide(probs, results[m]['threshold'])
    p, r, f = prf(y_dev, pred); rep = report(y_dev, pred)
    print(f'{m:30s} {results[m]["threshold"]:5.3f} {p:6.3f} {r:6.3f} {rep["micro_f1"]:9.4f} {rep["macro_f1"]:9.4f} {rep["samples_f1"]:10.4f}')

### 10.2 Precision / recall trade-off and threshold choice
Curves show how precision, recall and F1 change with the global threshold. Then **global vs per-label thresholds** are compared
fairly: dev is split in two halves, thresholds are tuned on one half and scored on the other (and swapped), so the per-label
thresholds are not just overfitting dev. Labels with fewer than 5 positives in the tuning half keep the global threshold.

In [ ]:
grid = np.arange(0.05, 0.95, 0.025)
fig, axes = plt.subplots(1, len(dev_probs), figsize=(4.2 * len(dev_probs), 3.4), squeeze=False)
for ax, (m, probs) in zip(axes[0], dev_probs.items()):
    ps, rs, fs = zip(*[prf(y_dev, decide(probs, t)) for t in grid])
    ax.plot(grid, ps, label='precision'); ax.plot(grid, rs, label='recall'); ax.plot(grid, fs, label='F1', lw=2)
    ax.set_title(m); ax.set_xlabel('threshold'); ax.grid(alpha=.3)
axes[0][0].legend(); plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'precision_recall_vs_threshold.png'), dpi=120); plt.show()

def tune_per_label(probs, y, g_thr, min_pos=5):
    thr = np.full(probs.shape[1], g_thr)
    for j in range(probs.shape[1]):
        if y[:, j].sum() < min_pos: continue
        f = [2 * (y[:, j] & (probs[:, j] >= t)).sum() / max(1, y[:, j].sum() + (probs[:, j] >= t).sum()) for t in grid]
        thr[j] = grid[int(np.argmax(f))]
    return thr

rng_cv = np.random.RandomState(SEED)
perm = rng_cv.permutation(len(y_dev)); folds = [perm[:len(perm) // 2], perm[len(perm) // 2:]]
print(f'{"method":30s} | {"global thr":>22s} | {"per-label thr":>22s}')
print(f'{"":30s} | {"micro-F1":>10s} {"macro-F1":>10s} | {"micro-F1":>10s} {"macro-F1":>10s}')
for m, probs in dev_probs.items():
    rows = {'g': [], 'l': []}
    for a, b in [(0, 1), (1, 0)]:
        tr, te = folds[a], folds[b]
        g, _ = best_threshold(probs[tr], y_dev[tr])
        per = tune_per_label(probs[tr], y_dev[tr], g)
        for key, pred in [('g', decide(probs[te], g)), ('l', decide(probs[te], per[None, :]))]:
            rep = report(y_dev[te], pred); rows[key].append((rep['micro_f1'], rep['macro_f1']))
    g_, l_ = np.mean(rows['g'], 0), np.mean(rows['l'], 0)
    print(f'{m:30s} | {g_[0]:10.4f} {g_[1]:10.4f} | {l_[0]:10.4f} {l_[1]:10.4f}')

### 10.3 Rare vs frequent statutes, and single- vs multi-statute documents
Statutes are bucketed by how many **training** documents mention them. Macro-F1 is computed inside each bucket
(only labels that appear in dev). Documents are bucketed by their number of gold statutes.

In [ ]:
buckets = {'rare (<10 train docs)': train_freq < 10,
           'medium (10-49)': (train_freq >= 10) & (train_freq < 50),
           'frequent (>=50)': train_freq >= 50}
print('statutes per bucket:', {k: int(v.sum()) for k, v in buckets.items()})
print(f'\n{"method":30s} ' + ' '.join(f'{k:>22s}' for k in buckets))
for m, probs in dev_probs.items():
    f1s, sup = per_label_f1(y_dev, decide(probs, results[m]['threshold']))
    cells_ = []
    for mask in buckets.values():
        use = mask & (sup > 0)
        cells_.append(f'{np.nanmean(f1s[use]):22.4f}' if use.any() else f'{"n/a":>22s}')
    print(f'{m:30s} ' + ' '.join(cells_))

n_gold = y_dev.sum(1)
doc_buckets = {'1 statute': n_gold == 1, '2-3 statutes': (n_gold >= 2) & (n_gold <= 3), '4+ statutes': n_gold >= 4}
print(f'\ndocuments per bucket: { {k: int(v.sum()) for k, v in doc_buckets.items()} }')
print(f'{"method":30s} ' + ' '.join(f'{k:>14s}' for k in doc_buckets) + '   (samples-F1 | avg #pred vs #gold)')
for m, probs in dev_probs.items():
    pred = decide(probs, results[m]['threshold'])
    out = []
    for mask in doc_buckets.values():
        out.append(f'{f1_score(y_dev[mask], pred[mask], average="samples", zero_division=0):14.4f}' if mask.any() else f'{"n/a":>14s}')
    print(f'{m:30s} ' + ' '.join(out) + f'   pred {pred.sum(1).mean():.2f} / gold {n_gold.mean():.2f}')

### 10.4 Label co-occurrence
Do the models reproduce which statutes tend to appear together? We list the most frequent statute pairs in train and
correlate the **gold** dev co-occurrence matrix with the **predicted** one (1.0 = same pattern).

In [ ]:
co = train_Y.T.astype(int) @ train_Y.astype(int)
np.fill_diagonal(co, 0)
top = np.dstack(np.unravel_index(np.argsort(-np.triu(co), axis=None)[:8], co.shape))[0]
print('most frequent statute pairs in train (statute IDs, #documents):')
for i, j in top:
    print(f'  ({label_ids[i]:>3d}, {label_ids[j]:>3d})  {co[i, j]}')

iu = np.triu_indices(N_LABELS, 1)
gold_co = (y_dev.T.astype(int) @ y_dev.astype(int))[iu]
print('\ncorrelation of predicted vs gold co-occurrence (dev):')
for m, probs in dev_probs.items():
    pr = decide(probs, results[m]['threshold'])
    pred_co = (pr.T.astype(int) @ pr.astype(int))[iu]
    c = np.corrcoef(gold_co, pred_co)[0, 1] if pred_co.std() > 0 and gold_co.std() > 0 else float('nan')
    print(f'  {m:30s} {c:.3f}')

### 10.5 Calibration
Expected calibration error (10 bins, over all document-statute probabilities) and Brier score, before and after a single
**temperature scaling** (`sigmoid(logit / T)`, T chosen on dev - slightly optimistic). Note that `pos_weight` deliberately
pushes probabilities up for rare statutes, so uncalibrated scores are expected.

In [ ]:
def ece(p, y, bins=10):
    p, y = p.ravel(), y.ravel().astype(float)
    edges = np.linspace(0, 1, bins + 1); tot = 0.0
    for lo, hi in zip(edges[:-1], edges[1:]):
        sel = (p >= lo) & (p < hi) if hi < 1 else (p >= lo)
        if sel.any():
            tot += sel.mean() * abs(p[sel].mean() - y[sel].mean())
    return tot

def temp_scale(p, T):
    z = np.log(np.clip(p, 1e-6, 1 - 1e-6) / (1 - np.clip(p, 1e-6, 1 - 1e-6)))
    return 1 / (1 + np.exp(-z / T))

def bce_np(p, y):
    p = np.clip(p, 1e-6, 1 - 1e-6); return -(y * np.log(p) + (1 - y) * np.log(1 - p)).mean()

print(f'{"method":30s} {"ECE":>7s} {"Brier":>7s} | {"best T":>6s} {"ECE":>7s} {"Brier":>7s} {"micro-F1 (re-tuned thr)":>24s}')
for m, probs in dev_probs.items():
    T = min(np.arange(0.5, 6.01, 0.1), key=lambda t: bce_np(temp_scale(probs, t), y_dev))
    cal = temp_scale(probs, T)
    thr_c, _ = best_threshold(cal, y_dev)
    print(f'{m:30s} {ece(probs, y_dev):7.4f} {((probs - y_dev) ** 2).mean():7.4f} | {T:6.1f} {ece(cal, y_dev):7.4f} '
          f'{((cal - y_dev) ** 2).mean():7.4f} {micro_f1(y_dev, decide(cal, thr_c)):24.4f}')

### 10.6 Where is the evidence? Window-attention analysis
For every **correctly predicted** (document, statute) pair in multi-window dev documents we look at the window with the highest attention for that statute:
- **peakedness**: its attention weight vs the uniform value `1/n_windows` (1.0x = attention is not selective, i.e. no better than mean pooling);
- **position**: whether the evidence window lies in the first, middle or last third of the document - if it is spread out, a head-only / truncated input must lose information;
- a few concrete examples (correct statute, missed statute, false positive) with the attention per window and the window's token offsets.

In [ ]:
att_runs = {}
for m in dev_probs:
    p = os.path.join(OUTPUT_DIR, f'dev_attn_{m}.pkl')
    if os.path.exists(p):
        att_runs[m] = pickle.load(open(p, 'rb'))

for m, A in att_runs.items():
    attn, starts = A['attn'], A['starts']
    pred = decide(dev_probs[m], results[m]['threshold'])
    peak, pos = [], []
    for i, a in enumerate(attn):
        n = a.shape[0]
        if n < 2: continue
        for j in np.where(y_dev[i] & pred[i])[0]:                        # true positives
            w = a[:, j].astype(float)
            peak.append(w.max() * n); pos.append(w.argmax() / (n - 1))
    if not peak:
        print(m, ': no multi-window true positives'); continue
    pos = np.array(pos)
    print(f'{m}: {len(peak)} correct (doc, statute) pairs in multi-window docs')
    print(f'   max attention weight = {np.mean(peak):.2f}x uniform (1.00x = not selective)')
    print(f'   evidence window in first third {np.mean(pos < 1/3):.1%} | middle {np.mean((pos >= 1/3) & (pos < 2/3)):.1%} | last third {np.mean(pos >= 2/3):.1%}')

    # a few concrete examples
    shown = {'correct': 0, 'missed': 0, 'false positive': 0}
    for i, a in enumerate(attn):
        if a.shape[0] < 3: continue
        for kind, js in (('correct', np.where(y_dev[i] & pred[i])[0]), ('missed', np.where(y_dev[i] & ~pred[i])[0]),
                         ('false positive', np.where(~y_dev[i] & pred[i])[0])):
            if len(js) and shown[kind] < 2:
                j = js[0]; shown[kind] += 1
                print(f'   [{kind}] doc {dev_ds[i]["id"]}, statute {label_ids[j]}: attention per window ' +
                      ' | '.join(f'{int(s)}:{w:.2f}' for s, w in zip(A['starts'][i], a[:, j].astype(float))))
        if all(v >= 2 for v in shown.values()): break

## 11. Observations (fill in after running)
Write 1-2 sentences per point using the tables above:
- **Best PEFT method** overall, and which method is best on **rare statutes** and on **multi-statute documents**.
- **Parameters vs accuracy:** how many trainable parameters each method needed (table in 10.1).
- **Thresholds:** did per-label thresholds beat a global one on held-out dev halves (10.2)? Where is the precision/recall balance?
- **Calibration:** was temperature scaling useful (10.5)?
- **Co-occurrence:** does the model reproduce frequent statute pairs (10.4)?
- **Window attention:** were the attention weights selective, and where in the document was the evidence (10.6)?

`prediction_<method>.json` is already in the required format. Submit the files of the best methods.